# Stage-1 Macro-Planning & Writing Utility Benchmark (Qwen2.5-7B-Instruct)
### Danh gia hai cap do (Group A: Intrinsic Plan Quality + VR-CLI | Group B: Writing Utility + Perturbation Sensitivity)

**Kien truc Benchmark Hai Nhanh (Two-Track Design)**:
1. **Track 1 — Nang luc Lap ke hoach (Planning Capacity via VR-CLI Next-Chapter Prediction)**:
   - Dataset: `LONGSTORY Test Split` (hoac `BookSum` / `PG-19` chia chuong chuan $y_{\text{gold}}$).
   - Chi so: $I_\pi$ (VR-CLI PPL Reduction %), $S_{\text{cov}}, S_{\text{prog}}$ (NLI), $S_{\text{causal}}$, **Story Arc Alignment ($S_{\text{arc}}$)**, va Planning Tokens.
2. **Track 2 — Hieu qua Sinh Truyen Dai & Ablation 5 Dieu kien (Conditions A-E)**:
   - Dataset: `TELL ME A STORY` (`tell-me-a-story-test.jsonl`, tu dong loc bo rang buoc do dai cu 900-1400 tu).
   - 5 Dieu kien: `A` (Direct) | `B` (Unstructured) | `C` (Linear Chapter) | `D` (Macro Event Graph Only) | `E` (Macro + Micro Sub-Events).
   - Chi so Group B: Word Count ($L$), $S_l$ (%), Early Stop Rate, DOC NLI Plan Adherence ($S_{\text{adh}}$), Distinct-3/4, Burstiness, va 5-Dimension LLM-Judge ($S_{\text{coh}}, C_{\text{weighted}}$).
3. **Perturbation Protocol (Kiem chung tinh le thuoc Dan y)**:
   - 5 nhieu chuan hoa: `causal_edge_removal`, `plot_order_inversion`, `motivation_stripping`, `unrelated_plan_swapping`, `context_conflict_injection`.

*Thiet ke an toan VRAM cho GPU 40GB (Colab A100 / Server H100): Phase 1 dung `vLLM` bfloat16 sinh toan bo dan y & truyen, giai phong GPU, sau do Phase 2 nap `Qwen2.5-7B` frozen (VR-CLI PPL) + `DeBERTa-v3 NLI`.*


## Buoc 1: Kiem tra GPU (A100 / H100 40GB) & Cai dat thu vien

In [ ]:
!nvidia-smi

import torch
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
    print("BF16 Supported:", torch.cuda.is_bf16_supported())


In [ ]:
# Cai dat vLLM va Transformers (khuyen nghi chay 1 lan khi khoi tao runtime)
%pip install -q vllm transformers accelerate sentencepiece datasets pandas


## Buoc 2: Ket noi Google Drive & Nap ma nguon MA-GRPO

In [ ]:
import os
import sys

# Bat Drive neu chay tren Google Colab de luu ket qua benchmark lau dai
USE_DRIVE = True
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTPUT_DIR = "/content/drive/MyDrive/MA-GRPO-MacroBenchmark"
    except Exception:
        OUTPUT_DIR = "./benchmark_outputs"
else:
    OUTPUT_DIR = "./benchmark_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output directory:", OUTPUT_DIR)

# Cap nhat duong dan repo tuy theo moi truong Colab hoac Server H100
REPO_URL = "https://github.com/AIVIETNAM-AIO-TruongDogki/MA_GRPO.git"
REPO_DIR = "/content/MA-GRPO"
if not os.path.exists(REPO_DIR) and not os.path.exists("src"):
    get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


## Buoc 3: Kiem thu Unit Tests he thong truoc khi chay GPU

In [ ]:
!python -m unittest discover -s tests -v


## Buoc 4: Cau hinh Dataset & Tuy chon Judge API Key
- `TRACK1_DATA`: Mac dinh dung `data/longstory_test.jsonl` (hoac doi sang `data/booksum_chapters.jsonl` cho tieu thuyet kinh dien BookSum).
- `TRACK2_DATA`: Mac dinh dung `data/tell-me-a-story-test.jsonl` (co the doi sang `data/mops_premises.jsonl` hoac `data/writingprompts_test.jsonl`).


In [ ]:
import getpass
import os

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
NLI_MODEL = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

# Duong dan dataset co san trong repo sau khi git pull
TRACK1_DATA = "data/longstory_test.jsonl"       # Hoac "data/booksum_chapters.jsonl"
TRACK2_DATA = "data/tell-me-a-story-test.jsonl" # Hoac "data/mops_premises.jsonl" / "data/writingprompts_test.jsonl"

# Tuy chon bat OpenAI / OpenRouter / Local vLLM Judge cho 5-Dimension Rubric & C_weighted
ENABLE_JUDGE = False
if ENABLE_JUDGE and not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Nhap OPENAI_API_KEY cho Judge (bo qua neu tat Judge): ")

print("Config ready -> Model:", MODEL_ID, "| NLI:", NLI_MODEL, "| Judge:", ENABLE_JUDGE)


## Buoc 5: Chay Benchmark Track 1 (VR-CLI Next-Chapter Prediction tren LONGSTORY)
*Chay cell nay khi ban da chi dinh `TRACK1_DATA` hop le.*


In [ ]:
import subprocess

if TRACK1_DATA and os.path.exists(TRACK1_DATA):
    cmd = [
        "python", "scripts/run_macro_planning_benchmark.py",
        "--mode", "track1",
        "--backend", "vllm",
        "--model", MODEL_ID,
        "--nli-model", NLI_MODEL,
        "--gpu-memory-utilization", "0.70",
        "--track1-data", TRACK1_DATA,
        "--track1-samples", "50",
        "--conditions", "B", "C", "D", "E",
        "--output-dir", OUTPUT_DIR,
        "--resume",
    ]
    subprocess.run(cmd, check=True)
else:
    print("Bo qua Track 1 vi chua chi dinh file TRACK1_DATA hop le. Dat duong dan LONGSTORY Test Split vao TRACK1_DATA de chay.")


## Buoc 6: Chay Track 2 (5-Condition Structural Ablation A-E) & Perturbation Audit
*Khuyen nghi chay pilot `--track2-samples 5` truoc de uoc luong toc do, sau do nang len `30` voi `--resume`.*


In [ ]:
import subprocess

cmd = [
    "python", "scripts/run_macro_planning_benchmark.py",
    "--mode", "track2",
    "--backend", "vllm",
    "--model", MODEL_ID,
    "--nli-model", NLI_MODEL,
    "--gpu-memory-utilization", "0.70",
    "--track2-samples", "10",   # Tang len 30 khi chay full paper benchmark
    "--target-words", "10000",
    "--num-chapters", "8",
    "--conditions", "A", "B", "C", "D", "E",
    "--output-dir", OUTPUT_DIR,
    "--resume",
]
if TRACK2_DATA:
    cmd.extend(["--track2-data", TRACK2_DATA])
if ENABLE_JUDGE:
    cmd.append("--enable-judge")

subprocess.run(cmd, check=True)


### Chay Perturbation Protocol (5 Bien the Nhieu tren Condition D)

In [ ]:
import subprocess

cmd_perturb = [
    "python", "scripts/run_macro_planning_benchmark.py",
    "--mode", "perturb",
    "--backend", "vllm",
    "--model", MODEL_ID,
    "--nli-model", NLI_MODEL,
    "--gpu-memory-utilization", "0.70",
    "--perturb-samples", "5",
    "--perturb-base-condition", "D",
    "--target-words", "10000",
    "--num-chapters", "8",
    "--output-dir", OUTPUT_DIR,
    "--resume",
]
if TRACK2_DATA:
    cmd_perturb.extend(["--track2-data", TRACK2_DATA])
if ENABLE_JUDGE:
    cmd_perturb.append("--enable-judge")

subprocess.run(cmd_perturb, check=True)


## Buoc 7: Hien thi Bang Ket qua CSV, Bao cao Markdown & Kiem dinh Gia thuyet H1-H4

In [ ]:
import json
import pandas as pd
from IPython.display import Markdown, display

for csv_name in ["track1_vrcli_results.csv", "track2_ablation_5conditions.csv", "perturbation_delta_results.csv"]:
    p = os.path.join(OUTPUT_DIR, csv_name)
    if os.path.exists(p):
        print(f"=== {csv_name} ===")
        display(pd.read_csv(p))

report_path = os.path.join(OUTPUT_DIR, "macro_planning_benchmark_report.md")
if os.path.exists(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))
